# 1. Objective and public API

This notebook defines a small, ordered state-vector layout prototype.

Its job is only to map semantic field/component names to integer indices, slices, units, and serializable layout metadata. It does not implement Kalman-filter math, covariance logic, process models, measurement models, estimator scheduling, plotting, quaternion algebra, or MRP algebra.

The intended composition is:

```text
StateVectorDefinition
    |-- VectorField
    |-- VectorField
    `-- AttitudeField
```

The order of supplied fields is the state-vector order:

```python
position = VectorField("position", ("x", "y", "z"), "m")
velocity = VectorField("velocity", ("vx", "vy", "vz"), "m/s")
attitude = AttitudeField("attitude", ("q1", "q2", "q3", "q4"), "quaternion")

state_definition = StateVectorDefinition([position, velocity, attitude])
```

This produces:

```text
x, y, z, vx, vy, vz, q1, q2, q3, q4
```

Attitude representation names are metadata here. Future representation classes should define what a quaternion, MRP, CRP, PRV, Euler angle set, or DCM means and should own representation-specific validation and operations.


## 2. Imports


In [1]:
from __future__ import annotations

from collections.abc import Sequence
from dataclasses import FrozenInstanceError, dataclass
from numbers import Integral
from types import MappingProxyType
from typing import Any, Protocol, runtime_checkable

import numpy as np


## 3. `StateField` protocol

`StateField` is a metadata contract. `VectorField` and `AttitudeField` satisfy it structurally.


In [2]:
@runtime_checkable
class StateField(Protocol):
    @property
    def name(self) -> str:
        ...

    @property
    def components(self) -> tuple[str, ...]:
        ...

    @property
    def units(self) -> tuple[str, ...]:
        ...

    @property
    def dimension(self) -> int:
        ...

    @property
    def field_type(self) -> str:
        ...


## 4. `AttitudeOperations` protocol

`AttitudeOperations` documents the dependency an `AttitudeField` may later receive.

The protocol declares the intended operations only. This notebook does not implement or call attitude mathematics.


In [3]:
@runtime_checkable
class AttitudeOperations(Protocol):
    @property
    def representation(self) -> str:
        ...

    @property
    def storage_dimension(self) -> int:
        ...

    def validate(self, value: np.ndarray) -> None:
        ...

    def normalize(self, value: np.ndarray) -> np.ndarray:
        ...

    def apply_increment(
        self,
        value: np.ndarray,
        increment: np.ndarray,
    ) -> np.ndarray:
        ...

    def difference(
        self,
        reference: np.ndarray,
        value: np.ndarray,
    ) -> np.ndarray:
        ...


## 5. Small validation helpers

Keep helpers small and boring. Field classes do their own domain validation inline.


In [4]:
def _require_text(value: Any, label: str) -> str:
    if not isinstance(value, str):
        raise TypeError(f"{label} must be a string.")
    if value.strip() == "":
        raise ValueError(f"{label} must be a non-empty string.")
    if value != value.strip():
        raise ValueError(f"{label} must not contain leading or trailing whitespace.")
    return value


def _normalize_units(units: str | Sequence[str], count: int, label: str) -> tuple[str, ...]:
    if isinstance(units, str):
        unit = _require_text(units, f"{label} unit")
        return tuple(unit for _ in range(count))

    unit_tuple = tuple(units)
    if len(unit_tuple) != count:
        raise ValueError(
            f"{label} units length must match component count {count}; "
            f"got {len(unit_tuple)}."
        )

    return tuple(
        _require_text(unit, f"{label} unit[{index}]")
        for index, unit in enumerate(unit_tuple)
    )


## 6. `VectorField`

`VectorField` is an immutable metadata object for ordinary vector components.


In [5]:
@dataclass(frozen=True)
class VectorField:
    name: str
    components: Sequence[str]
    units: str | Sequence[str]

    def __post_init__(self) -> None:
        name = _require_text(self.name, "VectorField name")
        components = tuple(self.components)

        if len(components) == 0:
            raise ValueError(f"VectorField '{name}' must define at least one component.")

        for index, component in enumerate(components):
            _require_text(component, f"VectorField '{name}' component[{index}]")

        if len(set(components)) != len(components):
            raise ValueError(f"VectorField '{name}' components must be unique.")

        object.__setattr__(self, "name", name)
        object.__setattr__(self, "components", components)
        object.__setattr__(
            self,
            "units",
            _normalize_units(self.units, len(components), f"VectorField '{name}'"),
        )

    @property
    def dimension(self) -> int:
        return len(self.components)

    @property
    def field_type(self) -> str:
        return "vector"


## 7. `AttitudeField`

`AttitudeField` records how attitude is stored in the state vector, but it does not define what each representation means.

Representation-specific classes should later own quaternion, MRP, CRP, PRV, Euler, and DCM validation. If an operations object is injected here, only its metadata compatibility is checked.


In [6]:
@dataclass(frozen=True)
class AttitudeField:
    name: str
    components: Sequence[str]
    representation: str
    units: str | Sequence[str] = "1"
    operations: AttitudeOperations | None = None

    def __post_init__(self) -> None:
        name = _require_text(self.name, "AttitudeField name")
        representation = _require_text(
            self.representation,
            f"AttitudeField '{name}' representation",
        ).lower()
        components = tuple(self.components)

        if len(components) == 0:
            raise ValueError(f"AttitudeField '{name}' must define at least one stored component.")

        for index, component in enumerate(components):
            _require_text(component, f"AttitudeField '{name}' component[{index}]")

        if len(set(components)) != len(components):
            raise ValueError(f"AttitudeField '{name}' components must be unique.")

        if self.operations is not None:
            self._validate_operations(self.operations, representation, len(components))

        object.__setattr__(self, "name", name)
        object.__setattr__(self, "components", components)
        object.__setattr__(self, "representation", representation)
        object.__setattr__(
            self,
            "units",
            _normalize_units(self.units, len(components), f"AttitudeField '{name}'"),
        )

    @property
    def dimension(self) -> int:
        return len(self.components)

    @property
    def field_type(self) -> str:
        return "attitude"

    @staticmethod
    def _validate_operations(
        operations: AttitudeOperations,
        representation: str,
        component_count: int,
    ) -> None:
        for attribute_name in ("representation", "storage_dimension"):
            if not hasattr(operations, attribute_name):
                raise TypeError(f"operations must provide '{attribute_name}'.")

        for method_name in ("validate", "normalize", "apply_increment", "difference"):
            if not callable(getattr(operations, method_name, None)):
                raise TypeError(f"operations must provide callable '{method_name}'.")

        operation_representation = _require_text(
            operations.representation,
            "operations representation",
        ).lower()
        if operation_representation != representation:
            raise ValueError(
                f"operations representation '{operation_representation}' does not match "
                f"field representation '{representation}'."
            )

        operation_dimension = operations.storage_dimension
        if isinstance(operation_dimension, bool) or not isinstance(operation_dimension, Integral):
            raise TypeError("operations storage_dimension must be an integer.")

        if int(operation_dimension) != component_count:
            raise ValueError(
                f"operations storage_dimension {operation_dimension} does not match "
                f"component count {component_count}."
            )


## 8. `StateVectorDefinition`

`StateVectorDefinition` resolves an ordered field sequence into immutable lookup tables.


In [7]:
@dataclass(frozen=True, init=False)
class StateVectorDefinition:
    _fields: tuple[StateField, ...]
    _dimension: int
    _field_names: tuple[str, ...]
    _component_names: tuple[str, ...]
    _component_indices: MappingProxyType
    _field_slices: MappingProxyType
    _component_units: MappingProxyType
    _field_units: MappingProxyType
    _field_lookup: MappingProxyType
    _field_types: MappingProxyType
    _field_representations: MappingProxyType

    def __init__(self, fields: Sequence[StateField]) -> None:
        field_tuple = tuple(fields)
        if len(field_tuple) == 0:
            raise ValueError("StateVectorDefinition requires at least one field.")

        field_names: list[str] = []
        component_names: list[str] = []
        component_indices: dict[str, int] = {}
        field_slices: dict[str, slice] = {}
        component_units: dict[str, str] = {}
        field_units: dict[str, tuple[str, ...]] = {}
        field_lookup: dict[str, StateField] = {}
        field_types: dict[str, str] = {}
        field_representations: dict[str, str | None] = {}

        start = 0
        for field in field_tuple:
            self._validate_field(field)

            if field.name in field_lookup:
                raise ValueError(f"Duplicate field name '{field.name}' is not allowed.")
            if field.name in component_indices:
                raise ValueError(
                    f"Field name '{field.name}' duplicates an existing component name."
                )

            for component in field.components:
                if component == field.name:
                    raise ValueError(
                        f"Field '{field.name}' contains a component with the same name."
                    )
                if component in field_lookup:
                    raise ValueError(
                        f"Component name '{component}' duplicates an existing field name."
                    )
                if component in component_indices:
                    raise ValueError(f"Duplicate component name '{component}' is not allowed.")

            stop = start + field.dimension
            field_names.append(field.name)
            field_slices[field.name] = slice(start, stop)
            field_units[field.name] = field.units
            field_lookup[field.name] = field
            field_types[field.name] = field.field_type
            field_representations[field.name] = getattr(field, "representation", None)

            for offset, component in enumerate(field.components):
                component_names.append(component)
                component_indices[component] = start + offset
                component_units[component] = field.units[offset]

            start = stop

        object.__setattr__(self, "_fields", field_tuple)
        object.__setattr__(self, "_dimension", start)
        object.__setattr__(self, "_field_names", tuple(field_names))
        object.__setattr__(self, "_component_names", tuple(component_names))
        object.__setattr__(self, "_component_indices", MappingProxyType(component_indices))
        object.__setattr__(self, "_field_slices", MappingProxyType(field_slices))
        object.__setattr__(self, "_component_units", MappingProxyType(component_units))
        object.__setattr__(self, "_field_units", MappingProxyType(field_units))
        object.__setattr__(self, "_field_lookup", MappingProxyType(field_lookup))
        object.__setattr__(self, "_field_types", MappingProxyType(field_types))
        object.__setattr__(self, "_field_representations", MappingProxyType(field_representations))

    def __len__(self) -> int:
        return self.dimension

    def __contains__(self, key: object) -> bool:
        return isinstance(key, str) and (
            key in self._component_indices or key in self._field_slices
        )

    def __getitem__(self, key: str) -> int | slice:
        if key in self._component_indices:
            return self._component_indices[key]
        if key in self._field_slices:
            return self._field_slices[key]
        raise KeyError(
            f"'{key}' is not a known component or field name. "
            f"Fields: {self.field_names}. Components: {self.component_names}."
        )

    def __repr__(self) -> str:
        pieces = []
        for field in self.fields:
            field_slice = self._field_slices[field.name]
            pieces.append(
                f"{field.name}[{field_slice.start}:{field_slice.stop}] "
                f"{field.field_type} {field.components} {field.units} "
                f"representation={self._field_representations[field.name]!r}"
            )
        return f"StateVectorDefinition(dimension={self.dimension}, fields={pieces})"

    @property
    def dimension(self) -> int:
        return self._dimension

    @property
    def field_names(self) -> tuple[str, ...]:
        return self._field_names

    @property
    def component_names(self) -> tuple[str, ...]:
        return self._component_names

    @property
    def fields(self) -> tuple[StateField, ...]:
        return self._fields

    def index(self, component_name: str) -> int:
        if component_name not in self._component_indices:
            raise KeyError(f"Unknown component name '{component_name}'.")
        return self._component_indices[component_name]

    def slice(self, field_name: str) -> slice:
        if field_name not in self._field_slices:
            raise KeyError(f"Unknown field name '{field_name}'.")
        return self._field_slices[field_name]

    def unit(self, component_name: str) -> str:
        if component_name not in self._component_units:
            raise KeyError(f"Unknown component name '{component_name}'.")
        return self._component_units[component_name]

    def units(self, field_name: str) -> tuple[str, ...]:
        if field_name not in self._field_units:
            raise KeyError(f"Unknown field name '{field_name}'.")
        return self._field_units[field_name]

    def field(self, field_name: str) -> StateField:
        if field_name not in self._field_lookup:
            raise KeyError(f"Unknown field name '{field_name}'.")
        return self._field_lookup[field_name]

    def contains_component(self, component_name: str) -> bool:
        return component_name in self._component_indices

    def contains_field(self, field_name: str) -> bool:
        return field_name in self._field_slices

    def get(self, values: np.ndarray, key: int | str) -> float | np.ndarray:
        self._validate_values(values)

        if isinstance(key, (bool, np.bool_)):
            raise TypeError("Boolean keys are not valid state-vector indices.")

        if isinstance(key, Integral):
            index = int(key)
            if index < 0 or index >= self.dimension:
                raise IndexError(
                    f"State index {index} is out of range for dimension {self.dimension}."
                )
            return values[index].item()

        resolved_key = self[key]
        if isinstance(resolved_key, slice):
            return values[resolved_key]
        return values[resolved_key].item()

    def describe(self) -> list[dict[str, object]]:
        rows: list[dict[str, object]] = []
        for field in self.fields:
            field_slice = self._field_slices[field.name]
            for offset, component in enumerate(field.components):
                rows.append(
                    {
                        "field_name": field.name,
                        "field_type": field.field_type,
                        "representation": self._field_representations[field.name],
                        "component_name": component,
                        "index": field_slice.start + offset,
                        "unit": field.units[offset],
                    }
                )
        return rows

    def to_spec(self) -> dict[str, object]:
        return {
            "dimension": self.dimension,
            "field_order": list(self.field_names),
            "component_order": list(self.component_names),
            "fields": [
                {
                    "name": field.name,
                    "field_type": field.field_type,
                    "components": list(field.components),
                    "units": list(field.units),
                    "start": self._field_slices[field.name].start,
                    "stop": self._field_slices[field.name].stop,
                    "representation": self._field_representations[field.name],
                }
                for field in self.fields
            ],
        }

    @staticmethod
    def _validate_field(field: StateField) -> None:
        for attribute_name in ("name", "components", "units", "dimension", "field_type"):
            if not hasattr(field, attribute_name):
                raise TypeError(f"State field must provide '{attribute_name}'.")

        if not isinstance(field.components, tuple):
            raise TypeError(f"Field '{field.name}' components must be stored as a tuple.")
        if not isinstance(field.units, tuple):
            raise TypeError(f"Field '{field.name}' units must be stored as a tuple.")
        if field.dimension != len(field.components):
            raise ValueError(
                f"Field '{field.name}' dimension must match its component count."
            )
        if len(field.units) != field.dimension:
            raise ValueError(f"Field '{field.name}' units must match its dimension.")

    def _validate_values(self, values: np.ndarray) -> None:
        if not isinstance(values, np.ndarray):
            raise TypeError("State values must be a NumPy ndarray.")
        if values.ndim != 1:
            raise ValueError(
                f"State values must be one-dimensional; got shape {values.shape}."
            )
        if values.size != self.dimension:
            raise ValueError(
                f"State values length must equal {self.dimension}; got {values.size}."
            )


## 9. Spacecraft-state example


In [8]:
position = VectorField(
    name="position",
    components=("x", "y", "z"),
    units="m",
)

velocity = VectorField(
    name="velocity",
    components=("vx", "vy", "vz"),
    units="m/s",
)

attitude = AttitudeField(
    name="attitude",
    components=("q1", "q2", "q3", "q4"),
    representation="quaternion",
)

state_definition = StateVectorDefinition([
    position,
    velocity,
    attitude,
])

mrp_attitude = AttitudeField(
    name="attitude",
    components=("sigma1", "sigma2", "sigma3"),
    representation="mrp",
)

state_definition


StateVectorDefinition(dimension=10, fields=["position[0:3] vector ('x', 'y', 'z') ('m', 'm', 'm') representation=None", "velocity[3:6] vector ('vx', 'vy', 'vz') ('m/s', 'm/s', 'm/s') representation=None", "attitude[6:10] attitude ('q1', 'q2', 'q3', 'q4') ('1', '1', '1', '1') representation='quaternion'"])

In [9]:
state_definition.dimension, state_definition.field_names, state_definition.component_names


(10,
 ('position', 'velocity', 'attitude'),
 ('x', 'y', 'z', 'vx', 'vy', 'vz', 'q1', 'q2', 'q3', 'q4'))

## 10. Index and slice access


In [10]:
(
    state_definition["x"],
    state_definition["position"],
    state_definition["q4"],
    state_definition["attitude"],
)


(0, slice(0, 3, None), 9, slice(6, 10, None))

In [11]:
(
    state_definition.index("vy"),
    state_definition.slice("velocity"),
    state_definition.unit("q1"),
    state_definition.units("position"),
    state_definition.field("attitude"),
)


(4,
 slice(3, 6, None),
 '1',
 ('m', 'm', 'm'),
 AttitudeField(name='attitude', components=('q1', 'q2', 'q3', 'q4'), representation='quaternion', units=('1', '1', '1', '1'), operations=None))

In [12]:
("x" in state_definition, "position" in state_definition, "attitude" in state_definition)


(True, True, True)

## 11. Numerical-state access


In [13]:
state_values = np.array([
    7000e3,
    0.0,
    0.0,
    0.0,
    7500.0,
    0.0,
    0.0,
    0.0,
    0.0,
    1.0,
])

(
    state_values[0],
    state_values[state_definition["x"]],
    state_values[state_definition["position"]],
    state_values[state_definition["attitude"]],
)


(7000000.0,
 7000000.0,
 array([7000000.,       0.,       0.]),
 array([0., 0., 0., 1.]))

In [14]:
(
    state_definition.get(state_values, 0),
    state_definition.get(state_values, "x"),
    state_definition.get(state_values, "position"),
    state_definition.get(state_values, "attitude"),
)


(7000000.0,
 7000000.0,
 array([7000000.,       0.,       0.]),
 array([0., 0., 0., 1.]))

## 12. Diagnostics and serialization

`describe()` returns plain rows for display. `to_spec()` returns language-neutral layout metadata and excludes operation objects.


In [15]:
state_definition.describe()


[{'field_name': 'position',
  'field_type': 'vector',
  'representation': None,
  'component_name': 'x',
  'index': 0,
  'unit': 'm'},
 {'field_name': 'position',
  'field_type': 'vector',
  'representation': None,
  'component_name': 'y',
  'index': 1,
  'unit': 'm'},
 {'field_name': 'position',
  'field_type': 'vector',
  'representation': None,
  'component_name': 'z',
  'index': 2,
  'unit': 'm'},
 {'field_name': 'velocity',
  'field_type': 'vector',
  'representation': None,
  'component_name': 'vx',
  'index': 3,
  'unit': 'm/s'},
 {'field_name': 'velocity',
  'field_type': 'vector',
  'representation': None,
  'component_name': 'vy',
  'index': 4,
  'unit': 'm/s'},
 {'field_name': 'velocity',
  'field_type': 'vector',
  'representation': None,
  'component_name': 'vz',
  'index': 5,
  'unit': 'm/s'},
 {'field_name': 'attitude',
  'field_type': 'attitude',
  'representation': 'quaternion',
  'component_name': 'q1',
  'index': 6,
  'unit': '1'},
 {'field_name': 'attitude',
  'fiel

In [16]:
state_definition.to_spec()


{'dimension': 10,
 'field_order': ['position', 'velocity', 'attitude'],
 'component_order': ['x', 'y', 'z', 'vx', 'vy', 'vz', 'q1', 'q2', 'q3', 'q4'],
 'fields': [{'name': 'position',
   'field_type': 'vector',
   'components': ['x', 'y', 'z'],
   'units': ['m', 'm', 'm'],
   'start': 0,
   'stop': 3,
   'representation': None},
  {'name': 'velocity',
   'field_type': 'vector',
   'components': ['vx', 'vy', 'vz'],
   'units': ['m/s', 'm/s', 'm/s'],
   'start': 3,
   'stop': 6,
   'representation': None},
  {'name': 'attitude',
   'field_type': 'attitude',
   'components': ['q1', 'q2', 'q3', 'q4'],
   'units': ['1', '1', '1', '1'],
   'start': 6,
   'stop': 10,
   'representation': 'quaternion'}]}

## 13. Validation examples


In [17]:
def show_expected_failure(description: str, fn) -> str:
    try:
        fn()
    except Exception as exc:
        return f"{description}: {type(exc).__name__}: {exc}"
    raise AssertionError(f"Expected failure did not occur: {description}")


expected_failures = [
    show_expected_failure(
        "duplicate component within one field",
        lambda: VectorField("bad", ("x", "x"), "m"),
    ),
    show_expected_failure(
        "empty attitude representation",
        lambda: AttitudeField("attitude", ("a", "b", "c"), ""),
    ),
    show_expected_failure(
        "unknown semantic lookup",
        lambda: state_definition["mass"],
    ),
]

expected_failures


["duplicate component within one field: ValueError: VectorField 'bad' components must be unique.",
 "empty attitude representation: ValueError: AttitudeField 'attitude' representation must be a non-empty string.",
 'unknown semantic lookup: KeyError: "\'mass\' is not a known component or field name. Fields: (\'position\', \'velocity\', \'attitude\'). Components: (\'x\', \'y\', \'z\', \'vx\', \'vy\', \'vz\', \'q1\', \'q2\', \'q3\', \'q4\')."']

## 14. Automated assertions


In [18]:
def assert_raises(expected_exception: type[BaseException], fn, contains: str | None = None) -> BaseException:
    try:
        fn()
    except expected_exception as exc:
        if contains is not None and contains.lower() not in str(exc).lower():
            raise AssertionError(
                f"Expected error message to contain {contains!r}; got {str(exc)!r}."
            ) from exc
        return exc
    except Exception as exc:
        raise AssertionError(
            f"Expected {expected_exception.__name__}; got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {expected_exception.__name__} to be raised.")


@dataclass(frozen=True)
class DummyAttitudeOperations:
    representation: str
    storage_dimension: int

    def validate(self, value: np.ndarray) -> None:
        return None

    def normalize(self, value: np.ndarray) -> np.ndarray:
        return value

    def apply_increment(self, value: np.ndarray, increment: np.ndarray) -> np.ndarray:
        return value

    def difference(self, reference: np.ndarray, value: np.ndarray) -> np.ndarray:
        return value - reference


# Ordered layout.
assert state_definition.field_names == ("position", "velocity", "attitude")
assert state_definition.component_names == (
    "x", "y", "z", "vx", "vy", "vz", "q1", "q2", "q3", "q4"
)
assert state_definition.dimension == 10
assert len(state_definition) == 10

# Component indices and field slices.
assert state_definition["x"] == 0
assert state_definition["q4"] == 9
assert state_definition["position"] == slice(0, 3)
assert state_definition["velocity"] == slice(3, 6)
assert state_definition["attitude"] == slice(6, 10)
assert state_definition.index("vy") == 4
assert state_definition.unit("q1") == "1"
assert state_definition.units("position") == ("m", "m", "m")
assert state_definition.field("attitude") is attitude

# Unit normalization.
assert VectorField("range", ("rho",), "m").units == ("m",)
assert VectorField("mixed", ("a", "b"), ("m", "s")).units == ("m", "s")
assert_raises(ValueError, lambda: VectorField("bad_units", ("x", "y"), ("m",)), "units length")
assert_raises(ValueError, lambda: VectorField("bad_unit", ("x",), ""), "non-empty")

# Field-level validation.
assert_raises(ValueError, lambda: VectorField("", ("x",), "m"), "non-empty")
assert_raises(ValueError, lambda: VectorField("empty", (), "m"), "at least one")
assert_raises(ValueError, lambda: VectorField("bad", ("x", "x"), "m"), "unique")
assert_raises(ValueError, lambda: AttitudeField("", ("q1",), "quaternion"), "non-empty")
assert_raises(ValueError, lambda: AttitudeField("attitude", (), "quaternion"), "at least one")
assert_raises(ValueError, lambda: AttitudeField("attitude", ("q1",), ""), "non-empty")

# Representation names are metadata in this notebook, not hardcoded math contracts.
custom_attitude = AttitudeField(
    name="attitude",
    components=("a", "b"),
    representation="FutureCustomRepresentation",
)
assert custom_attitude.representation == "futurecustomrepresentation"
assert custom_attitude.dimension == 2

# Global ambiguity checks.
assert_raises(
    ValueError,
    lambda: StateVectorDefinition([
        VectorField("position", ("x",), "m"),
        VectorField("position", ("y",), "m"),
    ]),
    "duplicate field",
)
assert_raises(
    ValueError,
    lambda: StateVectorDefinition([
        VectorField("a", ("x",), "m"),
        VectorField("b", ("x",), "m"),
    ]),
    "duplicate component",
)
assert_raises(
    ValueError,
    lambda: StateVectorDefinition([
        VectorField("position", ("x",), "m"),
        VectorField("x", ("vx",), "m/s"),
    ]),
    "duplicates an existing component",
)
assert_raises(
    ValueError,
    lambda: StateVectorDefinition([
        VectorField("position", ("position",), "m"),
    ]),
    "same name",
)

# Unknown lookup errors.
assert_raises(KeyError, lambda: state_definition["unknown"], "not a known")
assert_raises(KeyError, lambda: state_definition.index("unknown"), "unknown component")
assert_raises(KeyError, lambda: state_definition.slice("unknown"), "unknown field")
assert_raises(KeyError, lambda: state_definition.unit("unknown"), "unknown component")
assert_raises(KeyError, lambda: state_definition.units("unknown"), "unknown field")
assert_raises(KeyError, lambda: state_definition.field("unknown"), "unknown field")

# Numerical access validation.
assert_raises(TypeError, lambda: state_definition.get([1.0] * 10, "x"), "ndarray")
assert_raises(ValueError, lambda: state_definition.get(state_values.reshape(1, -1), "x"), "one-dimensional")
assert_raises(ValueError, lambda: state_definition.get(np.zeros(9), "x"), "length")
assert_raises(TypeError, lambda: state_definition.get(state_values, True), "boolean")
assert_raises(IndexError, lambda: state_definition.get(state_values, -1), "out of range")
assert_raises(IndexError, lambda: state_definition.get(state_values, 10), "out of range")

assert isinstance(state_definition.get(state_values, 0), float)
assert isinstance(state_definition.get(state_values, "x"), float)
assert state_definition.get(state_values, "x") == 7000e3
np.testing.assert_array_equal(
    state_definition.get(state_values, "position"),
    np.array([7000e3, 0.0, 0.0]),
)
np.testing.assert_array_equal(
    state_definition.get(state_values, "attitude"),
    np.array([0.0, 0.0, 0.0, 1.0]),
)

# Immutability.
assert_raises(FrozenInstanceError, lambda: setattr(position, "name", "new_position"))
assert_raises(FrozenInstanceError, lambda: setattr(state_definition, "_dimension", 11))
assert isinstance(state_definition.fields, tuple)

# Serialization order and operations exclusion.
spec = state_definition.to_spec()
assert spec["dimension"] == 10
assert spec["field_order"] == ["position", "velocity", "attitude"]
assert spec["component_order"] == ["x", "y", "z", "vx", "vy", "vz", "q1", "q2", "q3", "q4"]
assert [field["name"] for field in spec["fields"]] == ["position", "velocity", "attitude"]
assert spec["fields"][2]["representation"] == "quaternion"

operations = DummyAttitudeOperations("quaternion", 4)
attitude_with_operations = AttitudeField(
    "attitude",
    ("q1", "q2", "q3", "q4"),
    "quaternion",
    operations=operations,
)
operations_spec = StateVectorDefinition([attitude_with_operations]).to_spec()
assert "operations" not in operations_spec
assert all("operations" not in field_spec for field_spec in operations_spec["fields"])

assert_raises(
    ValueError,
    lambda: AttitudeField(
        "attitude",
        ("q1", "q2", "q3", "q4"),
        "quaternion",
        operations=DummyAttitudeOperations("quaternion", 3),
    ),
    "storage_dimension",
)
assert_raises(
    ValueError,
    lambda: AttitudeField(
        "attitude",
        ("q1", "q2", "q3", "q4"),
        "quaternion",
        operations=DummyAttitudeOperations("mrp", 4),
    ),
    "representation",
)

# Membership helpers.
assert state_definition.contains_component("x")
assert state_definition.contains_field("position")
assert "x" in state_definition
assert "position" in state_definition
assert "attitude" in state_definition
assert "unknown" not in state_definition

"All simplified automated assertions passed."


'All simplified automated assertions passed.'

## 15. Architectural conclusions and next notebook

This notebook establishes:

```text
semantic field definitions
ordered state-vector layout
vector and attitude field types
representation metadata
optional operations dependency injection
index and slice resolution
units metadata
validation
diagnostic descriptions
language-neutral serialization
```

The next notebook should introduce representation-specific classes and operations, beginning with:

```text
Euclidean vector operations
Quaternion attitude operations
```

Those classes should test:

```text
validation
normalization
apply_increment
difference
```

No Kalman filter is needed yet.
